<a href="https://colab.research.google.com/github/carlosmenendez10/Conversational-AI-Agent/blob/main/Conversational_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sqlite3
import pandas as pd
import random
from datetime import datetime, timedelta

# Conexion a SQLite
conn = sqlite3.connect('banca_sintetica.db')
cursor = conn.cursor()

# Estructura de las tablas
cursor.executescript('''
    DROP TABLE IF EXISTS movimientos;
    DROP TABLE IF EXISTS creditos;
    DROP TABLE IF EXISTS cuentas;
    DROP TABLE IF EXISTS clientes;

    CREATE TABLE clientes (
        id_cliente INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        segmento TEXT
    );

    CREATE TABLE cuentas (
        id_cuenta TEXT PRIMARY KEY,
        id_cliente INTEGER,
        tipo_producto TEXT,
        saldo_actual REAL,
        FOREIGN KEY(id_cliente) REFERENCES clientes(id_cliente)
    );

    CREATE TABLE creditos (
        id_credito TEXT PRIMARY KEY,
        id_cliente INTEGER,
        tipo_credito TEXT,
        etapa_riesgo INTEGER,
        saldo_insoluto REAL,
        dias_vencido INTEGER,
        FOREIGN KEY(id_cliente) REFERENCES clientes(id_cliente)
    );

    CREATE TABLE movimientos (
        id_movimiento INTEGER PRIMARY KEY AUTOINCREMENT,
        id_cuenta TEXT,
        fecha TEXT,
        concepto TEXT,
        categoria TEXT,
        monto REAL
    );
''')

# Datos Dummy
cursor.executemany("INSERT INTO clientes VALUES (?, ?, ?)", [
    (1, 'Ernesto Fernández', 'Select'),
    (2, 'Daniela Pérez', 'Clásico')
])

cursor.executemany("INSERT INTO cuentas VALUES (?, ?, ?, ?)", [
    ('TDC-1234', 1, 'Tarjeta de Crédito LikeU', 15000.00),
    ('DEB-5678', 1, 'Cuenta de Débito', 24500.50),
    ('TDC-9012', 2, 'Tarjeta de Crédito Clásica', 5000.00)
])

cursor.executemany("INSERT INTO creditos VALUES (?, ?, ?, ?, ?, ?)", [
    ('CRED-001', 1, 'Consumo', 1, 80000.00, 0),
    ('CRED-002', 2, 'Vivienda', 2, 1400000.00, 45)
])

# Generar algunos movimientos de prueba
movimientos_data = [
    ('TDC-1234', '2026-09-28', 'Sushiroll', 'Restaurantes', 450.00),
    ('TDC-1234', '2026-09-29', 'Estacionamiento', 'Transporte', 45.00),
    ('TDC-1234', '2026-09-30', 'Restaurante Somma', 'Restaurantes', 1200.00),
    ('DEB-5678', '2026-09-30', 'Nomina', 'Ingresos', 15000.00)
]
cursor.executemany("INSERT INTO movimientos (id_cuenta, fecha, concepto, categoria, monto) VALUES (?, ?, ?, ?, ?)", movimientos_data)
conn.commit()

print("Base de datos creada")

Base de datos creada


In [3]:
!pip install -q -U google-genai

from google import genai
from google.colab import userdata

# Configuración API
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

# Modelos disponibles
print("Modelos disponibles:")
for m in client.models.list():
    print(m.name)

# Prueba del modelo
respuesta = client.models.generate_content(
    model='gemini-3.5-flash',
    contents="Hola, dime en una oración qué es un Agente de Inteligencia Artificial."
)

print("\nPRUEBA DE RESPUESTA:")
print(respuesta.text)

Modelos disponibles:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flas

In [4]:
from google import genai
from google.colab import userdata

# Autenticación con cliente
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

# Modelos de Embeddings
print("Mis modelos de Embeddings disponibles son:")
for m in client.models.list():
    print(f"- {m.name}")

Mis modelos de Embeddings disponibles son:
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-4-26b-a4b-it
- models/gemma-4-31b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3.1-flash-lite
- models/gemini-3-pro-image-preview
- models/gemini-3-pro-image
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/gemini-3.1-flash-image
- models/gemini-3.1-flash-lite-image
- models/gemini-3.5-flash
- models/gemini-3.5-flash-lite
- models/gemini-omni-flash-preview
- models/gemini-omni-1.1-flash
- models/gemini-3.5-transcribe
- models/gemini-3.6-flash
- models/gemini-3.7-flash
- models/gemini-3.8-flash
- models/lyr

In [5]:
import requests
import numpy as np
from google.colab import userdata
from google import genai

# Autenticación con cliente
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

# Variables de configuración GitHub
usuario_github = "carlosmenendez10"
repositorio = "Conversational-AI-Agent"
rama = "main"
archivo = "politicas_banco.txt"

# Llave de GitHub (secreto)
github_token = userdata.get('GITHUB_TOKEN')

# URL del archivo
url_github = f"https://raw.githubusercontent.com/{usuario_github}/{repositorio}/{rama}/{archivo}"

# Conexion a github
headers = {'Authorization': f'token {github_token}'}

print("Conectando con GitHub...")
respuesta_github = requests.get(url_github, headers=headers)

if respuesta_github.status_code == 200:
    print("Archivo leído desde el repositorio")
    texto_leido = respuesta_github.text
else:
    print(f"Error al conectar con GitHub. Código: {respuesta_github.status_code}")
    print("Verifica tu usuario, el nombre del repo o tu token.")
    texto_leido = ""

# Motor RAG
if texto_leido:
    fragmentos = [frag.strip() for frag in texto_leido.split('\n\n') if frag.strip() != '']

    # 3. La función de embeddings actualizada al nuevo SDK
    def obtener_embedding(texto):
        respuesta = client.models.embed_content(
            model="gemini-embedding-2",
            contents=texto
        )
        return respuesta.embeddings[0].values

    print("Generando embeddings desde la nube...")
    embeddings_base = [obtener_embedding(frag) for frag in fragmentos]
    print("Motor RAG listo y conectado a GitHub")

    def buscar_informacion(pregunta_usuario):
        vector_pregunta = obtener_embedding(pregunta_usuario)
        similitudes = np.dot(embeddings_base, vector_pregunta)
        indice_mejor_resultado = np.argmax(similitudes)
        return fragmentos[indice_mejor_resultado]

    # PRUEBA DEL MOTOR RAG
    pregunta = "Necesito obligatoriamente un seguro de vida para mi hipoteca?"
    print(f"\nPregunta: {pregunta}")
    print(f"Contexto recuperado:\n{buscar_informacion(pregunta)}")

Conectando con GitHub...
Archivo leído desde el repositorio
Generando embeddings desde la nube...
Motor RAG listo y conectado a GitHub

Pregunta: Necesito obligatoriamente un seguro de vida para mi hipoteca?
Contexto recuperado:
SECCIÓN 2: CRÉDITOS DE VIVIENDA (HIPOTECARIOS)
Condiciones de Originación: Todos los créditos de vivienda requieren un enganche mínimo del 10% del valor del inmueble. El plazo máximo de financiamiento es de 20 años. Se requiere la contratación de un seguro de vida y daños vinculados al crédito.


In [6]:
########## API INTERACTIONS V2 #####################
import sqlite3
import requests
import numpy as np
from google import genai
from google.genai import types
from google.colab import userdata

# Inicializar el cliente (Embeddings)
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

# Descarga del txt desde github
usuario_github = "carlosmenendez10"
repositorio = "Conversational-AI-Agent"
rama = "main"
archivo = "politicas_banco.txt"
github_token = userdata.get('GITHUB_TOKEN')

url_github = f"https://raw.githubusercontent.com/{usuario_github}/{repositorio}/{rama}/{archivo}"
headers = {'Authorization': f'token {github_token}'}
respuesta_github = requests.get(url_github, headers=headers)
texto_leido = respuesta_github.text

# Chunking
fragmentos = [frag.strip() for frag in texto_leido.split('\n\n') if frag.strip() != '']

# Embeddings
def obtener_embedding_nuevo(texto):
    respuesta = client.models.embed_content(
        model='gemini-embedding-2', # Usamos el modelo estándar de embeddings
        contents=texto
    )

    return respuesta.embeddings[0].values

print("Generando embeddings con el nuevo SDK...")
embeddings_base = [obtener_embedding_nuevo(frag) for frag in fragmentos]

# Tools y manejo de errores
def consultar_base_datos(consulta_sql: str) -> str:
    """
    Ejecuta una consulta SQL en la base de datos del Banco y devuelve el resultado.
    Úsala EXCLUSIVAMENTE cuando el usuario pregunte por sus transacciones, saldos, productos o movimientos.

    Tablas disponibles:
    - clientes (id_cliente, nombre, segmento)
    - cuentas (id_cuenta, id_cliente, tipo_producto, saldo_actual)
    - creditos (id_credito, id_cliente, tipo_credito, etapa_riesgo, saldo_insoluto, dias_vencido)
    - movimientos (id_movimiento, id_cuenta, fecha, concepto, categoria, monto)

    REGLAS FUNDAMENTALES (SE DEBEN SEGUIR ESTRICTAMENTE):
    1. "Productos" significa tanto lo que hay en la tabla 'cuentas' como en la tabla 'creditos'.
    2. Si te preguntan por los productos de un cliente, debes consultar AMBAS tablas (con UNION o en consultas separadas).
    3. NUNCA inventes productos, créditos, cuentas ni saldos. Solo menciona EXACTAMENTE lo que el resultado SQL te devuelva. CERO alucinaciones.
    4. NUNCA proporcionar datos de otros clientes. Solo datos que tengan el ID de cliente proporcionado
    """
    print(f"\n[DEBUG SQL] La IA generó esta consulta: {consulta_sql}")
    try:
        conn = sqlite3.connect('banca_sintetica.db')
        cursor = conn.cursor()
        cursor.execute(consulta_sql)
        resultados = cursor.fetchall()
        conn.close()
        print(f"[DEBUG DE SQL] Resultado de la base de datos: {resultados}")
        return str(resultados)
    except Exception as e:
        print(f"\n[DEBUG DESQL] ¡ERROR EN CONSULTA!: {e}")
        return f"Error SQL: {e}"

def consultar_politicas(pregunta_usuario: str) -> str:
    """
    Busca en las políticas internas del Banco y devuelve el fragmento más relevante.
    Úsala EXCLUSIVAMENTE cuando el usuario pregunte por reglas, anualidades, penalizaciones o beneficios.
    """
    print(f"\n[DEBUG RAG] La IA decidió buscar esta frase: '{pregunta_usuario}'")

    try:
        # Embeddng de pregunta
        vector_pregunta = obtener_embedding_nuevo(pregunta_usuario)

        # Similitud del Coseno (Mayor -> temas similares)
        A = np.array(embeddings_base)
        B = np.array(vector_pregunta)

        normas_A = np.linalg.norm(A, axis=1)
        norma_B = np.linalg.norm(B)
        similitudes = np.dot(A, B) / (normas_A * norma_B)

        # Se obtiene el mejor resultado
        indice_mejor = np.argmax(similitudes)
        fragmento_elegido = fragmentos[indice_mejor]

        print(f"[DEBUG RAG] Fragmento recuperado: {fragmento_elegido[:75]}...")
        return fragmento_elegido

    except Exception as e:
        # Si algo falla
        print(f"\n[DEBUG RAG] ERROR: {e}")
        return f"Dile al cliente que hubo un error técnico interno: {e}"

# 5 Agente con la API (se actualiza a interactions en lugar de generateContent)
chat = client.chats.create(
    model="gemini-3.5-flash-lite",
    config=types.GenerateContentConfig(
        system_instruction="Eres un asesor financiero experto del Banco. Sé amable, claro y directo.",
        tools=[consultar_base_datos, consultar_politicas],
        temperature=0.2, # Temperatura (creatividad)
    )
)

# 6. Prueba Final
mensaje_cliente = """
Hola. Quiero saber algunas cosas:
Mi número de cliente (ID) es 1.
1. Alguna d mis compras tiene beneficios como generar cashback? En dado caso cuánto sería aproximadamente?
"""

print("El Agente está pensando y gestionando herramientas en la nube...\n")
respuesta_agente = chat.send_message(mensaje_cliente)

print("--- RESPUESTA FINAL DEL AGENTE ---")
print(respuesta_agente.text)

Generando embeddings con el nuevo SDK...
El Agente está pensando y gestionando herramientas en la nube...


[DEBUG SQL] La IA generó esta consulta: SELECT tipo_producto, saldo_actual FROM cuentas WHERE id_cliente = 1 UNION SELECT tipo_credito, saldo_insoluto FROM creditos WHERE id_cliente = 1
[DEBUG DE SQL] Resultado de la base de datos: [('Consumo', 80000.0), ('Cuenta de Débito', 24500.5), ('Tarjeta de Crédito LikeU', 15000.0)]

[DEBUG RAG] La IA decidió buscar esta frase: 'generar cashback compras beneficio'
[DEBUG RAG] Fragmento recuperado: SECCIÓN 1: TARJETAS DE CRÉDITO
Producto Tarjeta de Crédito Especial: Exclus...

[DEBUG SQL] La IA generó esta consulta: SELECT * FROM movimientos WHERE id_cuenta IN (SELECT id_cuenta FROM cuentas WHERE id_cliente = 1)
[DEBUG DE SQL] Resultado de la base de datos: [(1, 'TDC-1234', '2026-09-28', 'Sushiroll', 'Restaurantes', 450.0), (2, 'TDC-1234', '2026-09-29', 'Estacionamiento', 'Transporte', 45.0), (3, 'TDC-1234', '2026-09-30', 'Restaurante Somma